# Updated Phase 1 — Task 4 (Shristi) · Baseline + Loss + Training Skeleton

**MirrorSpeech** — Colab-ready notebook aligned with **Task 1** (L2-ARCTIC + LibriSpeech), **Task 2** (paths + Whisper), and **Task 3** (5 accent classes).

**Before running:** **Runtime → Change runtime type → GPU** (T4 or better). Whisper zero-shot + BERTScore are slow on CPU.

**Data scope:** Evaluates the **full test split** from Task 1 (L2-ARCTIC ± LibriSpeech per your `test.json`). To evaluate **L2 accents only**, filter `test_records` to `accent_id in (0,1,2,3)` before building `test_loader`.

**Course focus — SER (Sentence Error Rate):** Your professor wants the team to **treat SER seriously**. SER is the **fraction of test utterances** where the recognized text is **not exactly equal** to the reference after shared normalization (same rule as in this notebook’s `normalize_text`). **Lower SER is better.** Any single wrong word makes the whole sentence count as an error, so SER is often **stricter / higher** than WER alone suggests — improving **WER/CER** through training usually **pulls SER down** too.

**Also reported:** **WER**, **CER**, **PER**, **WIL**, **BERTScore F1** — per accent and overall.

It does the following:

1. Mounts Google Drive and prepares **L2-ARCTIC** (from zips) and optionally **LibriSpeech** (`dev-clean`)
2. Loads `train.json`, `val.json`, `test.json`, and `config.json` from `MirrorSpeech/splits/`
3. Fixes `wav_path` the same way as Task 2 (L2 under `/content/data/l2arctic/`, Libri under Drive or `/content/data/librispeech/`)
4. Builds dataloaders
5. Runs **Whisper-small zero-shot** baseline on the test set
6. Computes **WER, CER, SER, PER, WIL, BERTScore F1** (per accent + overall)
7. Saves **`baseline_metrics.json`**
8. Implements **Total Loss = ASR Loss + α × Accent Loss + β × LCMA Loss**
9. Runs a **1-batch training skeleton** with **`num_accent_classes`** from `config.json` (typically **5**)
10. Saves a checkpoint under **`MyDrive/MirrorSpeech/checkpoints/`** and runs evaluation

---

## Expected JSON record format

```python
{
    "wav_path": "/path/to/audio.wav",   # or .flac for LibriSpeech
    "transcript": "text transcript here",
    "accent_id": 0,
    "speaker": "ABA"                    # or "librispeech_1234" for native
}
```

Accent IDs follow Task 1 / `config.json` (usually **0–3** = L2 accents, **4** = native / LibriSpeech).


In [1]:
# =========================================
# Step 0 — Install required packages
# =========================================
!pip install -q torch torchaudio transformers jiwer bert-score g2p_en peft accelerate soundfile


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.3/180.3 kB 5.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 98.7 MB/s eta 0:00:00


In [2]:
# =========================================
# Step 1 — Mount Google Drive + extract L2-ARCTIC
# =========================================
# Same layout as Updated Task 2: zips in MirrorSpeech/l2arctic_zips/

from google.colab import drive
import os, zipfile, json

drive.mount('/content/drive')

DRIVE_ZIP_DIR = '/content/drive/MyDrive/MirrorSpeech/l2arctic_zips'
EXTRACT_PATH  = '/content/data/l2arctic'
SPLITS_DIR    = '/content/drive/MyDrive/MirrorSpeech/splits'
L2ARCTIC_PATH = '/content/data/l2arctic'

SPEAKERS = ['RRBI','SVBI','TNI','NJS',
            'HQTV','MBMPS','NCC','TXHC',
            'HJK','HKK','YDCK','YKWK',
            'ABA','YBAA','SKA','ZHAA']

os.makedirs(EXTRACT_PATH, exist_ok=True)

for spk in SPEAKERS:
    zip_path = f'{DRIVE_ZIP_DIR}/{spk}.zip'
    out_path = f'{EXTRACT_PATH}/{spk}'
    if os.path.exists(out_path):
        print(f'  ⏭️  {spk} already extracted')
        continue
    if not os.path.exists(zip_path):
        print(f'  ❌  {spk}.zip not found in Drive')
        continue
    print(f'  Extracting {spk}...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(EXTRACT_PATH)

print(f'\n✅ L2-ARCTIC speakers under {EXTRACT_PATH}')


Mounted at /content/drive
  Extracting RRBI...
  Extracting SVBI...
  Extracting TNI...
  Extracting NJS...
  Extracting HQTV...
  Extracting MBMPS...
  Extracting NCC...
  Extracting TXHC...
  Extracting HJK...
  Extracting HKK...
  Extracting YDCK...
  Extracting YKWK...
  Extracting ABA...
  Extracting YBAA...
  Extracting SKA...
  Extracting ZHAA...

✅ L2-ARCTIC speakers under /content/data/l2arctic


In [3]:
# =========================================
# Step 1b — Optional: LibriSpeech dev-clean (if not on Drive)
# =========================================
# If native rows fail to load, run this once; Task 2 uses the same layout.

import torchaudio
LIBRISPEECH_PATH = '/content/data/librispeech'
LIBRISPEECH_SPLIT = 'dev-clean'
os.makedirs(LIBRISPEECH_PATH, exist_ok=True)
print(f"Downloading LibriSpeech '{LIBRISPEECH_SPLIT}' → {LIBRISPEECH_PATH} (~344 MB, one-time per fresh runtime)...")
_libri = torchaudio.datasets.LIBRISPEECH(root=LIBRISPEECH_PATH, url=LIBRISPEECH_SPLIT, download=True)
_root = os.path.join(LIBRISPEECH_PATH, 'LibriSpeech', LIBRISPEECH_SPLIT)
print(f"✅ LibriSpeech ready: {len(_libri):,} utterances | on disk: {os.path.isdir(_root)}")


100%|██████████| 322M/322M [00:17<00:00, 18.9MB/s]


✅ LibriSpeech ready: 2,703 utterances | on disk: True


In [4]:
# =========================================
# Step 2 — Load splits + fix wav_path (L2 + LibriSpeech)
# =========================================
# Mirrors Updated_Phase1_Task2_Vidushi_Colab.ipynb — Cell 2

from collections import Counter

DATA_ROOT = os.path.dirname(L2ARCTIC_PATH)  # e.g. /content/data

def load_and_fix_paths(split_name):
    """L2: .../l2arctic/{speaker}/wav/{file}. Libri: Drive MirrorSpeech/librispeech/... or /content/data/librispeech/..."""
    path    = f'{SPLITS_DIR}/{split_name}.json'
    records = json.load(open(path))
    mirror_root = os.path.dirname(SPLITS_DIR)
    for r in records:
        orig = r['wav_path']
        norm = orig.replace('\\', '/')
        if 'librispeech' in norm.lower():
            tail = norm.split('librispeech/', 1)[-1]
            drive_libri = os.path.join(mirror_root, 'librispeech', tail)
            if norm.startswith('data/'):
                content_libri = os.path.join('/content', norm)
            else:
                content_libri = os.path.join(DATA_ROOT, norm.split('data/', 1)[-1]) if 'data/' in norm else drive_libri
            if os.path.isfile(drive_libri):
                r['wav_path'] = drive_libri
            elif os.path.isfile(content_libri):
                r['wav_path'] = content_libri
            else:
                r['wav_path'] = drive_libri
        else:
            r['wav_path'] = os.path.join(
                L2ARCTIC_PATH,
                r['speaker'],
                'wav',
                os.path.basename(orig),
            )
    return records

with open(f'{SPLITS_DIR}/config.json', 'r') as f:
    config = json.load(f)

train_records = load_and_fix_paths('train')
val_records   = load_and_fix_paths('val')
test_records  = load_and_fix_paths('test')

ACCENT_MAP = {int(k): v for k, v in config['id_to_accent'].items()}
NUM_ACCENT_CLASSES = int(config.get('num_accent_classes', len(ACCENT_MAP)))

print('=== PROJECT DATASET ===')
print(f'Train : {len(train_records):,} samples')
print(f'Val   : {len(val_records):,} samples')
print(f'Test  : {len(test_records):,} samples')
print(f'num_accent_classes: {NUM_ACCENT_CLASSES}')
print('\nAccent classes (train):')
for aid, cnt in sorted(Counter(r['accent_id'] for r in train_records).items()):
    print(f'  {ACCENT_MAP[aid]:10s} (id={aid}): {cnt:,}')
print(f"\nSample L2 path : {train_records[0]['wav_path']}")
print(f"File exists    : {os.path.exists(train_records[0]['wav_path'])}")
_ex = next((r for r in train_records if r.get('accent_id') == 4), None)
if _ex:
    print(f"\nNative / Libri : {_ex['wav_path']}")
    print(f"File exists    : {os.path.isfile(_ex['wav_path'])}")
    if not os.path.isfile(_ex['wav_path']):
        print('  → Put Libri under Drive: MyDrive/MirrorSpeech/librispeech/ or run Step 1b.')


=== PROJECT DATASET ===
Train : 14,829 samples
Val   : 1,853 samples
Test  : 1,855 samples
num_accent_classes: 5

Accent classes (train):
  indian     (id=0): 3,599
  mandarin   (id=1): 3,649
  korean     (id=2): 2,728
  arabic     (id=3): 2,686
  native     (id=4): 2,167

Sample L2 path : /content/data/l2arctic/NJS/wav/arctic_b0385.wav
File exists    : True

Native / Libri : /content/drive/MyDrive/MirrorSpeech/librispeech/LibriSpeech/dev-clean/2803/161169/2803-161169-0006.flac
File exists    : True


In [5]:
# =========================================
# Step 3 — Imports and global settings
# =========================================

import re
from collections import defaultdict

import torch
import torchaudio
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from jiwer import cer, wil, wer
from bert_score import score as bertscore_score

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
TARGET_SR = 16000

# Consistent accent order for metrics tables
ACCENT_ORDER = [ACCENT_MAP[i] for i in sorted(ACCENT_MAP.keys())]

print('Using device:', DEVICE)
print('Accent order for reporting:', ACCENT_ORDER)


Using device: cuda
Accent order for reporting: ['indian', 'mandarin', 'korean', 'arabic', 'native']


In [6]:
# =========================================
# Step 4 — Load Whisper processor
# =========================================

processor = WhisperProcessor.from_pretrained('openai/whisper-small')
print('Whisper processor loaded successfully.')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Whisper processor loaded successfully.


## Metrics used for evaluation

1. **WER (Word Error Rate)** — Word-level edit distance between reference and hypothesis after normalization (`jiwer.wer`).
2. **CER (Character Error Rate)** — Edit distance (insertions, deletions, substitutions) at character level vs reference length (`jiwer.cer`).
3. **SER (Sentence Error Rate)** — **Sentence-level** score: fraction of utterances where `normalize_text(hypothesis) != normalize_text(reference)` (strict **exact string** match). One small mistake → whole sentence counts as wrong, so SER is a **hard** barometer of “did we get the full sentence exactly right?” **Training goal:** reduce SER vs the Whisper baseline on the **same** test split and normalization. SER is not a differentiable loss in this notebook; you improve it **indirectly** by better ASR (real token loss, fine-tuning, accent-aware training, etc.).
4. **PER (Phoneme Error Rate)** — Edit distance between reference and hypothesis **phoneme sequences**; here we use `g2p_en` to approximate phone strings from text (replace with gold phonemes if your data provides them).
5. **WIL (Word Information Lost)** — Word-level information loss via alignment (`jiwer.wil`).
6. **BERTScore (F1)** — Semantic similarity using contextual BERT embeddings: F1 over token-level recall and precision (`bert-score`); rewards paraphrases and equivalent wording, not only exact string match.


In [7]:
# =========================================
# Step 5 — Metric helper functions
# =========================================

def normalize_text(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"[^\w\s']", ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def sentence_error_rate(refs, hyps) -> float:
    """SER: fraction of pairs where normalized hyp != normalized ref (exact match).
    Used for course / professor reporting — lower is better."""
    if len(refs) == 0:
        return 0.0
    wrong = sum(1 for r, h in zip(refs, hyps) if normalize_text(r) != normalize_text(h))
    return wrong / len(refs)


_g2p_singleton = None

def _get_g2p():
    global _g2p_singleton
    if _g2p_singleton is None:
        from g2p_en import G2p
        _g2p_singleton = G2p()
    return _g2p_singleton

def text_to_phones(text: str) -> str:
    try:
        g2p = _get_g2p()
        phones = [p for p in g2p(normalize_text(text)) if str(p).strip() and p != ' ']
        return ' '.join(map(str, phones))
    except Exception:
        return ' '.join(list(normalize_text(text).replace(' ', '')))


def phoneme_error_rate(refs, hyps) -> float:
    if len(refs) == 0:
        return 0.0
    ref_phones = [text_to_phones(x) for x in refs]
    hyp_phones = [text_to_phones(x) for x in hyps]
    return wer(ref_phones, hyp_phones)


def compute_all_metrics(refs, hyps):
    refs = [normalize_text(x) for x in refs]
    hyps = [normalize_text(x) for x in hyps]
    metrics = {
        'wer': wer(refs, hyps) if refs else 0.0,
        'cer': cer(refs, hyps) if refs else 0.0,
        'ser': sentence_error_rate(refs, hyps),
        'per': phoneme_error_rate(refs, hyps),
        'wil': wil(refs, hyps) if refs else 0.0,
        'bertscore_f1': 0.0,
    }
    if refs:
        _, _, f1 = bertscore_score(hyps, refs, lang='en', verbose=False)
        metrics['bertscore_f1'] = float(f1.mean().item())
    return metrics

print('Metric helpers ready.')


Metric helpers ready.


In [8]:
# =========================================
# Step 6 — Dataset and DataLoader (L2 .wav + Libri .flac)
# =========================================

class MirrorSpeechWhisperDataset(Dataset):
    def __init__(self, records, processor):
        self.records = records
        self.processor = processor

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        record = self.records[idx]
        waveform, sr = torchaudio.load(record['wav_path'])
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        if sr != TARGET_SR:
            waveform = torchaudio.transforms.Resample(sr, TARGET_SR)(waveform)
        audio = waveform.squeeze(0).numpy()
        input_features = self.processor(
            audio,
            sampling_rate=TARGET_SR,
            return_tensors='pt'
        ).input_features.squeeze(0)
        return {
            'input_features': input_features,
            'transcript': record['transcript'],
            'accent_id': int(record['accent_id']),
            'speaker': record.get('speaker', 'unknown'),
            'wav_path': record['wav_path'],
        }


def collate_fn(batch):
    return {
        'input_features': torch.stack([x['input_features'] for x in batch]),
        'transcript': [x['transcript'] for x in batch],
        'accent_id': torch.tensor([x['accent_id'] for x in batch], dtype=torch.long),
        'speaker': [x['speaker'] for x in batch],
        'wav_path': [x['wav_path'] for x in batch],
    }


train_dataset = MirrorSpeechWhisperDataset(train_records, processor)
val_dataset = MirrorSpeechWhisperDataset(val_records, processor)
test_dataset = MirrorSpeechWhisperDataset(test_records, processor)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, collate_fn=collate_fn)

print('Train batches:', len(train_loader))
print('Val batches  :', len(val_loader))
print('Test batches :', len(test_loader))


Train batches: 3708
Val batches  : 464
Test batches : 464


In [9]:
# =========================================
# Step 7 — Sanity check one batch
# =========================================

sample_batch = next(iter(train_loader))
print('input_features shape:', sample_batch['input_features'].shape)
print('num transcripts     :', len(sample_batch['transcript']))
print('accent_id shape     :', sample_batch['accent_id'].shape)
print('first transcript    :', sample_batch['transcript'][0])
print('first accent_id     :', sample_batch['accent_id'][0].item(), '→', ACCENT_MAP[int(sample_batch['accent_id'][0])])
print('first wav_path      :', sample_batch['wav_path'][0])


input_features shape: torch.Size([4, 80, 3000])
num transcripts     : 4
accent_id shape     : torch.Size([4])
first transcript    : But what they want with your toothbrush is more than I can imagine
first accent_id     : 0 → indian
first wav_path      : /content/data/l2arctic/RRBI/wav/arctic_a0285.wav


## Baseline: Whisper-small zero-shot

Runs **Whisper-small** on the test set with **no fine-tuning**. Results are saved to **`baseline_metrics.json`** (WER, CER, SER, PER, WIL, BERTScore F1).


In [10]:
# =========================================
# Step 8 — Load Whisper-small baseline model
# =========================================

baseline_model = WhisperForConditionalGeneration.from_pretrained('openai/whisper-small').to(DEVICE)
baseline_model.eval()
print('Baseline Whisper-small loaded.')


model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

Baseline Whisper-small loaded.


In [11]:
# =========================================
# Step 9 — Zero-shot baseline (per accent + overall)
# =========================================

@torch.no_grad()
def run_zero_shot_baseline(model, dataloader, processor, output_path='baseline_metrics.json'):
    refs_by_accent = defaultdict(list)
    hyps_by_accent = defaultdict(list)

    for batch_idx, batch in enumerate(dataloader):
        feats = batch['input_features'].to(DEVICE)
        refs = batch['transcript']
        accent_ids = batch['accent_id']
        pred_ids = model.generate(feats)
        hyps = processor.batch_decode(pred_ids, skip_special_tokens=True)
        for ref, hyp, aid in zip(refs, hyps, accent_ids):
            name = ACCENT_MAP[int(aid)]
            refs_by_accent[name].append(ref)
            hyps_by_accent[name].append(hyp)
        if batch_idx % 25 == 0:
            print(f'Processed baseline batch {batch_idx + 1}/{len(dataloader)}')

    results = {}
    all_refs, all_hyps = [], []
    for accent in ACCENT_ORDER:
        refs = refs_by_accent[accent]
        hyps = hyps_by_accent[accent]
        metrics = compute_all_metrics(refs, hyps)
        results[accent] = {
            'wer': round(metrics['wer'], 4),
            'cer': round(metrics['cer'], 4),
            'ser': round(metrics['ser'], 4),
            'per': round(metrics['per'], 4),
            'wil': round(metrics['wil'], 4),
            'bertscore_f1': round(metrics['bertscore_f1'], 4),
            'num_samples': len(refs),
        }
        all_refs.extend(refs)
        all_hyps.extend(hyps)

    overall = compute_all_metrics(all_refs, all_hyps)
    results['overall'] = {
        'model': 'openai/whisper-small',
        'mode': 'zero-shot',
        'wer': round(overall['wer'], 4),
        'cer': round(overall['cer'], 4),
        'ser': round(overall['ser'], 4),
        'per': round(overall['per'], 4),
        'wil': round(overall['wil'], 4),
        'bertscore_f1': round(overall['bertscore_f1'], 4),
        'num_samples': len(all_refs),
    }

    with open(output_path, 'w') as f:
        json.dump(results, f, indent=2)
    print('\nMetrics per accent (WER / CER / SER / PER / WIL / BERT F1):')
    for accent in ACCENT_ORDER:
        if accent in results:
            m = results[accent]
            print(f"  {accent:12s}  WER={m['wer']:.4f}  CER={m['cer']:.4f}  SER={m['ser']:.4f}  PER={m['per']:.4f}  WIL={m['wil']:.4f}  BERT={m['bertscore_f1']:.4f}  (n={m['num_samples']})")
    o = results['overall']
    print(f"  {'overall':12s}  WER={o['wer']:.4f}  CER={o['cer']:.4f}  SER={o['ser']:.4f}  PER={o['per']:.4f}  WIL={o['wil']:.4f}  BERT={o['bertscore_f1']:.4f}  (n={o['num_samples']})")
    print('\n=== SER summary (course / professor focus: lower SER = more exact sentence matches) ===')
    for accent in ACCENT_ORDER:
        if accent in results:
            s = results[accent]['ser']
            n = results[accent]['num_samples']
            print(f"  {accent:12s}  SER={s:.4f}   exact-match sentences: {100*(1-s):.1f}%   (n={n})")
    print(f"  {'overall':12s}  SER={o['ser']:.4f}   exact-match sentences: {100*(1-o['ser']):.1f}%   (n={o['num_samples']})")
    print('\nFull baseline results (JSON):')
    print(json.dumps(results, indent=2))
    print(f'\nSaved baseline metrics to: {output_path}')
    return results


baseline_results = run_zero_shot_baseline(
    baseline_model,
    test_loader,
    processor,
    output_path='baseline_metrics.json'
)


Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.log

Processed baseline batch 1/464
Processed baseline batch 26/464
Processed baseline batch 51/464
Processed baseline batch 76/464
Processed baseline batch 101/464
Processed baseline batch 126/464
Processed baseline batch 151/464
Processed baseline batch 176/464
Processed baseline batch 201/464
Processed baseline batch 226/464
Processed baseline batch 251/464
Processed baseline batch 276/464
Processed baseline batch 301/464
Processed baseline batch 326/464
Processed baseline batch 351/464
Processed baseline batch 376/464
Processed baseline batch 401/464
Processed baseline batch 426/464
Processed baseline batch 451/464


[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package cmudict to /root/nltk_data...
[nltk_data]   Unzipping corpora/cmudict.zip.


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Metrics per accent (WER / CER / SER / PER / WIL / BERT F1):
  indian        WER=0.0866  CER=0.0384  SER=0.4420  PER=0.0422  WIL=0.1508  BERT=0.9787  (n=457)
  mandarin      WER=0.1849  CER=0.0995  SER=0.6452  PER=0.1084  WIL=0.3014  BERT=0.9583  (n=451)
  korean        WER=0.1106  CER=0.0530  SER=0.5612  PER=0.0583  WIL=0.1916  BERT=0.9734  (n=335)
  arabic        WER=0.8685  CER=0.7311  SER=0.7118  PER=0.7476  WIL=0.7479  BERT=0.8973  (n=340)
  native        WER=0.0455  CER=0.0177  SER=0.4632  PER=0.0193  WIL=0.0800  BERT=0.9870  (n=272)
  overall       WER=0.2187  CER=0.1529  SER=0.5655  PER=0.1596  WIL=0.2992  BERT=0.9591  (n=1855)

Full baseline results (JSON):
{
  "indian": {
    "wer": 0.0866,
    "cer": 0.0384,
    "ser": 0.442,
    "per": 0.0422,
    "wil": 0.1508,
    "bertscore_f1": 0.9787,
    "num_samples": 457
  },
  "mandarin": {
    "wer": 0.1849,
    "cer": 0.0995,
    "ser": 0.6452,
    "per": 0.1084,
    "wil": 0.3014,
    "bertscore_f1": 0.9583,
    "num_samples": 4

## Training skeleton for Task 4

Proof that the **training loop** is wired: forward → combined loss → backward → checkpoint.

The **ASR loss** below remains a **placeholder scalar** (same as the original Task 4 skeleton) so the notebook stays stable before full ASR integration. **Accent classification** uses **`NUM_ACCENT_CLASSES`** (5 with LibriSpeech native).


In [12]:
# =========================================
# Step 10 — Task 4 model components (aligned with Tasks 2–3)
# =========================================

class ContentHead(nn.Module):
    def __init__(self, input_dim=768, output_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, output_dim),
            nn.LayerNorm(output_dim),
            nn.ReLU()
        )

    def forward(self, x):
        return self.net(x)


class AccentHead(nn.Module):
    def __init__(self, input_dim=768, accent_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, accent_dim),
            nn.LayerNorm(accent_dim),
            nn.ReLU()
        )

    def forward(self, encoder_output):
        return self.net(encoder_output).mean(dim=1)


class AccentClassifier(nn.Module):
    def __init__(self, accent_dim=128, num_accents=None):
        num_accents = num_accents if num_accents is not None else NUM_ACCENT_CLASSES
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(accent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, num_accents)
        )

    def forward(self, accent_vec):
        return self.net(accent_vec)


class Reencoder(nn.Module):
    def __init__(self, input_dim=384, output_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, output_dim),
            nn.LayerNorm(output_dim),
            nn.ReLU()
        )

    def forward(self, x):
        return self.net(x)


In [13]:
# =========================================
# Step 11 — Combined loss
# =========================================
# Total Loss = ASR Loss + α × Accent Loss + β × LCMA Loss

class CombinedLoss(nn.Module):
    def __init__(self, alpha=1.0, beta=1.0):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.accent_loss_fn = nn.CrossEntropyLoss()
        self.lcma_loss_fn = nn.MSELoss()

    def forward(self, asr_loss, accent_logits, accent_labels, original_content, reencoded_content):
        accent_loss = self.accent_loss_fn(accent_logits, accent_labels)
        lcma_loss = self.lcma_loss_fn(original_content, reencoded_content)
        total_loss = asr_loss + self.alpha * accent_loss + self.beta * lcma_loss
        return {
            'total_loss': total_loss,
            'asr_loss': asr_loss,
            'accent_loss': accent_loss,
            'lcma_loss': lcma_loss,
        }

criterion = CombinedLoss(alpha=1.0, beta=1.0)
print('CombinedLoss initialized.')


CombinedLoss initialized.


In [14]:
# =========================================
# Step 12 — Task 4 model wrapper
# =========================================

class MirrorSpeechTask4(nn.Module):
    def __init__(self, whisper_model, content_head, accent_head, accent_classifier, reencoder):
        super().__init__()
        self.whisper_model = whisper_model
        self.encoder = whisper_model.model.encoder
        self.content_head = content_head
        self.accent_head = accent_head
        self.accent_classifier = accent_classifier
        self.reencoder = reencoder

    def forward(self, input_features, accent_labels=None):
        enc_out = self.encoder(input_features).last_hidden_state
        content = self.content_head(enc_out)
        accent_vec = self.accent_head(enc_out)
        accent_logits = self.accent_classifier(accent_vec)
        b, t, _ = content.shape
        neutral_accent = torch.zeros(b, t, 128, device=content.device)
        swapped = torch.cat([content, neutral_accent], dim=-1)
        reencoded_content = self.reencoder(swapped)
        asr_loss = torch.tensor(1.0, device=content.device, requires_grad=True)
        return {
            'asr_loss': asr_loss,
            'accent_logits': accent_logits,
            'content_vectors': content,
            'reencoded_content_vectors': reencoded_content,
        }

    def generate(self, input_features):
        return self.whisper_model.generate(input_features)


In [15]:
# =========================================
# Step 13 — Initialize Task 4 model
# =========================================

task4_whisper = WhisperForConditionalGeneration.from_pretrained('openai/whisper-small')

content_head = ContentHead()
accent_head = AccentHead()
accent_classifier = AccentClassifier(num_accents=NUM_ACCENT_CLASSES)
reencoder = Reencoder()

task4_model = MirrorSpeechTask4(
    whisper_model=task4_whisper,
    content_head=content_head,
    accent_head=accent_head,
    accent_classifier=accent_classifier,
    reencoder=reencoder,
).to(DEVICE)

print('Task 4 model initialized (accent classifier outputs:', NUM_ACCENT_CLASSES, 'classes).')


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Task 4 model initialized (accent classifier outputs: 5 classes).


In [16]:
# =========================================
# Step 14 — One-batch training skeleton + checkpoint on Drive
# =========================================

CKPT_DIR = '/content/drive/MyDrive/MirrorSpeech/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)

CKPT_PATH = f'{CKPT_DIR}/task4_phase1.pt'
LOCAL_CKPT = 'checkpoints/task4_demo.pt'

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, task4_model.parameters()),
    lr=1e-4
)

task4_model.train()
for epoch in range(1, 2):
    for batch_idx, batch in enumerate(train_loader):
        input_features = batch['input_features'].to(DEVICE)
        accent_labels = batch['accent_id'].to(DEVICE)
        optimizer.zero_grad()
        outputs = task4_model(input_features=input_features, accent_labels=accent_labels)
        loss_dict = criterion(
            asr_loss=outputs['asr_loss'],
            accent_logits=outputs['accent_logits'],
            accent_labels=accent_labels,
            original_content=outputs['content_vectors'],
            reencoded_content=outputs['reencoded_content_vectors']
        )
        loss_dict['total_loss'].backward()
        optimizer.step()
        print('=== TASK 4 TRAINING SKELETON ===')
        print(f'Epoch        : {epoch}')
        print(f'Batch        : {batch_idx}')
        print(f"total_loss   : {loss_dict['total_loss'].item():.4f}")
        print(f"asr_loss     : {loss_dict['asr_loss'].item():.4f}")
        print(f"accent_loss  : {loss_dict['accent_loss'].item():.4f}")
        print(f"lcma_loss    : {loss_dict['lcma_loss'].item():.4f}")
        torch.save(task4_model.state_dict(), LOCAL_CKPT)
        torch.save(task4_model.state_dict(), CKPT_PATH)
        print(f'\nSaved checkpoint: {LOCAL_CKPT}')
        print(f'Saved checkpoint: {CKPT_PATH}')
        break


=== TASK 4 TRAINING SKELETON ===
Epoch        : 1
Batch        : 0
total_loss   : 3.2541
asr_loss     : 1.0000
accent_loss  : 1.5978
lcma_loss    : 0.6563

Saved checkpoint: checkpoints/task4_demo.pt
Saved checkpoint: /content/drive/MyDrive/MirrorSpeech/checkpoints/task4_phase1.pt


In [17]:
# =========================================
# Step 15 — Evaluate checkpoint (per accent + overall)
# =========================================

@torch.no_grad()
def evaluate_checkpoint(model, processor, dataloader, checkpoint_path, output_path='task4_eval_results.json'):
    model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE), strict=False)
    model.to(DEVICE)
    model.eval()
    refs_by_accent = defaultdict(list)
    hyps_by_accent = defaultdict(list)
    for batch_idx, batch in enumerate(dataloader):
        feats = batch['input_features'].to(DEVICE)
        refs = batch['transcript']
        accent_ids = batch['accent_id']
        pred_ids = model.generate(feats)
        hyps = processor.batch_decode(pred_ids, skip_special_tokens=True)
        for ref, hyp, aid in zip(refs, hyps, accent_ids):
            name = ACCENT_MAP[int(aid)]
            refs_by_accent[name].append(ref)
            hyps_by_accent[name].append(hyp)
        if batch_idx % 25 == 0:
            print(f'Processed eval batch {batch_idx + 1}/{len(dataloader)}')

    results = {}
    all_refs, all_hyps = [], []
    for accent in ACCENT_ORDER:
        refs = refs_by_accent[accent]
        hyps = hyps_by_accent[accent]
        metrics = compute_all_metrics(refs, hyps)
        results[accent] = {
            'wer': round(metrics['wer'], 4),
            'cer': round(metrics['cer'], 4),
            'ser': round(metrics['ser'], 4),
            'per': round(metrics['per'], 4),
            'wil': round(metrics['wil'], 4),
            'bertscore_f1': round(metrics['bertscore_f1'], 4),
            'num_samples': len(refs),
        }
        all_refs.extend(refs)
        all_hyps.extend(hyps)
    overall = compute_all_metrics(all_refs, all_hyps)
    results['overall'] = {
        'wer': round(overall['wer'], 4),
        'cer': round(overall['cer'], 4),
        'ser': round(overall['ser'], 4),
        'per': round(overall['per'], 4),
        'wil': round(overall['wil'], 4),
        'bertscore_f1': round(overall['bertscore_f1'], 4),
        'num_samples': len(all_refs),
    }
    with open(output_path, 'w') as f:
        json.dump(results, f, indent=2)
    print('\nMetrics per accent (WER / CER / SER / PER / WIL / BERT F1):')
    for accent in ACCENT_ORDER:
        if accent in results and accent != 'overall':
            m = results[accent]
            print(f"  {accent:12s}  WER={m['wer']:.4f}  CER={m['cer']:.4f}  SER={m['ser']:.4f}  PER={m['per']:.4f}  WIL={m['wil']:.4f}  BERT={m['bertscore_f1']:.4f}  (n={m['num_samples']})")
    o = results['overall']
    print(f"  {'overall':12s}  WER={o['wer']:.4f}  CER={o['cer']:.4f}  SER={o['ser']:.4f}  PER={o['per']:.4f}  WIL={o['wil']:.4f}  BERT={o['bertscore_f1']:.4f}  (n={o['num_samples']})")
    print('\n=== SER summary (course / professor focus: lower SER = more exact sentence matches) ===')
    for accent in ACCENT_ORDER:
        if accent in results and accent != 'overall':
            s = results[accent]['ser']
            n = results[accent]['num_samples']
            print(f"  {accent:12s}  SER={s:.4f}   exact-match sentences: {100*(1-s):.1f}%   (n={n})")
    print(f"  {'overall':12s}  SER={o['ser']:.4f}   exact-match sentences: {100*(1-o['ser']):.1f}%   (n={o['num_samples']})")
    print('\nFull evaluation results (JSON):')
    print(json.dumps(results, indent=2))
    print(f'\nSaved evaluation metrics to: {output_path}')
    return results


eval_results = evaluate_checkpoint(
    model=task4_model,
    processor=processor,
    dataloader=test_loader,
    checkpoint_path=LOCAL_CKPT,
    output_path='task4_eval_results.json'
)


Processed eval batch 1/464
Processed eval batch 26/464
Processed eval batch 51/464
Processed eval batch 76/464
Processed eval batch 101/464
Processed eval batch 126/464
Processed eval batch 151/464
Processed eval batch 176/464
Processed eval batch 201/464
Processed eval batch 226/464
Processed eval batch 251/464
Processed eval batch 276/464
Processed eval batch 301/464
Processed eval batch 326/464
Processed eval batch 351/464
Processed eval batch 376/464
Processed eval batch 401/464
Processed eval batch 426/464
Processed eval batch 451/464


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Metrics per accent (WER / CER / SER / PER / WIL / BERT F1):
  indian        WER=0.0844  CER=0.0381  SER=0.4420  PER=0.0420  WIL=0.1478  BERT=0.9796  (n=457)
  mandarin      WER=0.2144  CER=0.1161  SER=0.6430  PER=0.1218  WIL=0.3211  BERT=0.9590  (n=451)
  korean        WER=0.1189  CER=0.0581  SER=0.5731  PER=0.0635  WIL=0.2043  BERT=0.9722  (n=335)
  arabic        WER=1.0917  CER=0.8970  SER=0.6824  PER=0.8885  WIL=0.7439  BERT=0.9134  (n=340)
  native        WER=0.0461  CER=0.0182  SER=0.4779  PER=0.0199  WIL=0.0812  BERT=0.9868  (n=272)
  overall       WER=0.2600  CER=0.1824  SER=0.5639  PER=0.1849  WIL=0.3186  BERT=0.9622  (n=1855)

Full evaluation results (JSON):
{
  "indian": {
    "wer": 0.0844,
    "cer": 0.0381,
    "ser": 0.442,
    "per": 0.042,
    "wil": 0.1478,
    "bertscore_f1": 0.9796,
    "num_samples": 457
  },
  "mandarin": {
    "wer": 0.2144,
    "cer": 0.1161,
    "ser": 0.643,
    "per": 0.1218,
    "wil": 0.3211,
    "bertscore_f1": 0.959,
    "num_samples": 45

## Final outputs

After a full run you should have:

| File | Description |
|------|-------------|
| `baseline_metrics.json` | Zero-shot Whisper-small: WER, CER, SER, PER, WIL, BERTScore F1 |
| `checkpoints/task4_demo.pt` | Local checkpoint copy |
| `MyDrive/MirrorSpeech/checkpoints/task4_phase1.pt` | Persistent checkpoint on Drive |
| `task4_eval_results.json` | Eval metrics after loading the skeleton checkpoint |

Together these show **Task 4** deliverables: baseline, combined loss, training skeleton, checkpoint, evaluation — on the **L2-ARCTIC + LibriSpeech** splits from Phase 1.
